# Image Classifier

In this notebook we will be designing a simple classifier network to Classify Images of Cats and Fish 

## Preparing Dataset

We will load the dataset using pytorch.  
After loading the images we will transform all the images to be of size 64x64.  
Then the image values are transformed into tensors.  
Then we normalize all the channels using standard Imagenet preprocessing values

In [162]:
import torch
import torchvision
from torchvision import transforms
from torch.utils import data

train_data_path="./Fish-vs-Cats/train/"
val_data_path="./Fish-vs-Cats/val/"
test_data_path="./Fish-vs-Cats/test/"

transforms= transforms.Compose([
    transforms.Resize((64,64)),          #Resize the image to 64x64
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229, 0.224, 0.225])
])

train_data=torchvision.datasets.ImageFolder(root=train_data_path,transform=transforms)
val_data=torchvision.datasets.ImageFolder(root=val_data_path,transform=transforms)
test_data=torchvision.datasets.ImageFolder(root=test_data_path,transform=transforms)

batch_size=64
train_data_loader= data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
val_data_loader= data.DataLoader(val_data, batch_size=batch_size, shuffle=True)
test_data_loader= data.DataLoader(test_data, batch_size=batch_size, shuffle=True)

## Design of the Neural Network

Here we design our neural network.  
-  The tensors of our images are of the shape 3x64x64. We need to reduce the dimension in order to feed them into our neural network. There for we flatten our input first. Then the neural network contains 4 fully connected layers with Relu layers in between them.  
-  Relu layers introduce non-linearities between linear linear transformations enabling network to learn complex, non-linear patterns. 
-  We will apply softmax to the output

In [163]:
from torch import nn
import torch.nn.functional as f

class SimpleNet(nn.Module):

    def __init__(self):
        super(SimpleNet,self).__init__()
        self.flatten = nn.Flatten()
        self.relu= nn.ReLU()
        self.fc1 = nn.Linear(12288,2048)
        self.fc2 = nn.Linear(2048,256)
        self.fc3 = nn.Linear(256,32)
        self.fc4 = nn.Linear(32,2)
        

    def forward(self,x):
        x=self.flatten(x)
        x=self.fc1(x)
        x=self.relu(x)
        x=self.fc2(x)
        x=self.relu(x)
        x=self.fc3(x)
        x=self.relu(x)
        x=self.fc4(x)
        return x
    




In [ ]:
if torch.cuda.is_available():
    device=torch.device("cuda")
else:
    device=torch.device("cpu")

## Model Training

We will train our model for the dataset to tune the parameters of neural network.  
- Loss function - We are using the CrossEntropy Loss as the loss function which is usually preffered for classification network.

- Optimizer - After testing for some time I have selected Adam based on the performance. SGD seemed to finding it hard to converge. Adam showed overfitting for the default lr.  

This combinational still is not the best. I got accuracy of around 0.78 on validation set. Can try out changing the parameters and optimizers for better performance



In [ ]:
import torch.optim as optim
from tqdm import tqdm



def train_model(model,optimizer,loss_fn,train_loader,val_loader,epochs=20,device='cpu'):
    best_val_loss = float('inf')
    best_model=None
    
    for epoch in range(epochs):
        train_loss = 0.0
        val_loss = 0.0
        model.train()
        for batch in tqdm(train_loader):
            optimizer.zero_grad()
            inputs,target=batch
            inputs=inputs.to(device)
            target=target.to(device)
            output = model (inputs)
            loss = loss_fn (output,target)
            loss.backward()
            optimizer.step()
            train_loss+=loss.data.item()*inputs.size(0)
        train_loss/=len(train_loader.dataset)

        model.eval()
        num_correct=0
        num_examples=0
        with torch.no_grad():
            for batch in tqdm(val_loader):
                inputs,target=batch
                inputs=inputs.to(device)
                target=target.to(device)
                output = model (inputs)
                loss = loss_fn (output,target)
                val_loss+=loss.data.item()*inputs.size(0)
                predicted = torch.argmax(output, dim=1)
                num_correct += (predicted == target).sum().item()
                num_examples += len(target)
        val_loss/=len(val_loader.dataset)
        accuracy = num_correct/num_examples


        print('Epoch: {} , Training loss: {:.2f}, Validation loss: {:.2f}, accuracy: {:.2f}'.format(epoch,train_loss,val_loss,accuracy))


model = SimpleNet()
model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.00001)
epochs = 20

In [167]:
train_model(model,optimizer,loss_fn,train_data_loader,val_data_loader,epochs=20,device=device)

100%|██████████| 2/2 [00:00<00:00,  5.85it/s]


Epoch: 0 , Training loss: 0.66, Validation loss: 0.64, accuracy: 0.66


100%|██████████| 2/2 [00:00<00:00,  6.16it/s]


Epoch: 1 , Training loss: 0.60, Validation loss: 0.63, accuracy: 0.69


100%|██████████| 2/2 [00:00<00:00,  5.56it/s]


...

Epoch: 3 , Training loss: 0.51, Validation loss: 0.58, accuracy: 0.75


100%|██████████| 2/2 [00:00<00:00,  6.49it/s]


Epoch: 4 , Training loss: 0.48, Validation loss: 0.55, accuracy: 0.73


 77%|███████▋  | 10/13 [00:02<00:00,  4.95it/s]


KeyboardInterrupt: 

Testing the model on a sample image

In [ ]:
from PIL import Image

labels = ['cat','fish']
img = Image.open("Fish-vs-Cats/test/fish/1406240463_a50c959b4f.jpg")
img = transforms(img)
img = img.unsqueeze(0)
img=img.to(device)
model.eval()
pred = model(img)
print(labels[pred.argmax()])

fish


Saving the parameters of the model so that it can be used later.  
```
torch.save(model,PATH)
```
can also be used. But then we won't be abale to use it if we change the structure of the model later. So saving the state dictionary is the better option.

In [ ]:
torch.save(model.state_dict(),"models/simplenet")   #saving the model params